# 07 — Pull FAO Food Price Index

**Source:** FAO Food Price Monitoring and Analysis (FPMA)  
**Provider:** Food and Agriculture Organization of the United Nations  
**Access:** FAOSTAT API (no key required) + direct CSV download  
**Coverage:** Global, 1961–present  
**Frequency:** Monthly (Food Price Indices); Annual (country-level food inflation via FAOSTAT)

## What this notebook pulls

| Table | Description | Why it matters |
|---|---|---|
| **FAO Food Price Index (FFPI)** | Global monthly indices (Food, Cereals, Meat, Dairy, Sugar, Oils) | #1 SHAP predictor in IMF Barrett et al. (2021) social unrest model |
| **FAOSTAT country food prices** | Domestic food price level per country, annual | Country-level food inflation as structural predictor |

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

In [ ]:
import os
import io
import zipfile
import requests
import pandas as pd
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

# Global FAO Food Price Index — CSV published monthly by FAO
FFPI_URL = "https://www.fao.org/3/cb8726en/cb8726en.xlsx"
# Fallback: direct data API
FAOSTAT_API_BASE  = "https://fenixservices.fao.org/faostat/api/v1/en"

# FAOSTAT element code for Consumer Price Index of food (element 6176)
# Domain: Prices (CP), Item: Food (item code 23011)
FAOSTAT_DOMAIN    = "CP"   # Consumer Prices
FAOSTAT_ELEMENT   = "6176" # CPI of food

print(f"Run date: {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## 1. Global FAO Food Price Index (FFPI)

The FFPI is a monthly global commodity price index. We pull it from the FAO
statistical publication Excel file, then fall back to the FAOSTAT API if the
static URL is no longer valid.

Base period: 2014–2016 = 100.

In [ ]:
FFPI_COMPONENT_NAMES = {
    "Food Price Index": "ffpi_food",
    "Meat Price Index": "ffpi_meat",
    "Dairy Price Index": "ffpi_dairy",
    "Cereals Price Index": "ffpi_cereals",
    "Oils Price Index": "ffpi_oils",
    "Sugar Price Index": "ffpi_sugar",
}

def pull_ffpi_from_excel() -> pd.DataFrame | None:
    print(f"Downloading FFPI Excel from {FFPI_URL} ...")
    try:
        resp = requests.get(FFPI_URL, timeout=60)
        resp.raise_for_status()
    except Exception as e:
        print(f"  Failed: {e}")
        return None

    # The FFPI spreadsheet has a header; actual data starts on a variable row
    df_raw = pd.read_excel(io.BytesIO(resp.content), sheet_name=0, header=None)

    # Find the row that contains 'Year' or a recognisable date header
    header_row = None
    for i, row in df_raw.iterrows():
        if any(str(v).strip().lower() in ("year", "month") for v in row.values if pd.notna(v)):
            header_row = i
            break

    if header_row is None:
        print("  Could not find header row in FFPI Excel")
        return None

    df = pd.read_excel(io.BytesIO(resp.content), sheet_name=0, header=header_row)
    df.columns = [str(c).strip() for c in df.columns]
    df.dropna(how="all", inplace=True)
    return df


def pull_ffpi_from_faostat_api() -> pd.DataFrame:
    """Fallback: pull FFPI via the FAOSTAT bulk download API."""
    print("Pulling FFPI via FAOSTAT API...")
    # FAOSTAT domain FP = Food Price Index
    url = f"{FAOSTAT_API_BASE}/data/FP"
    params = {
        "area": "5000",  # World aggregate
        "element": "F1",
        "year": ",".join(str(y) for y in range(1990, datetime.utcnow().year + 1)),
        "output_type": "csv",
    }
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    return pd.read_csv(io.BytesIO(resp.content))


df_ffpi_raw = pull_ffpi_from_excel()
if df_ffpi_raw is None or df_ffpi_raw.empty:
    df_ffpi_raw = pull_ffpi_from_faostat_api()

print(f"FFPI raw shape: {df_ffpi_raw.shape}")
df_ffpi_raw.head(3)

In [ ]:
# Tidy FFPI: the Excel format is typically wide with year + month rows and
# one column per index component. Melt to long format.
df_ffpi = df_ffpi_raw.copy()

# Normalise column names
df_ffpi.columns = [str(c).strip() for c in df_ffpi.columns]

# Identify year and month columns
year_col  = next((c for c in df_ffpi.columns if c.lower() == "year"),  None)
month_col = next((c for c in df_ffpi.columns if c.lower() == "month"), None)

if year_col and month_col:
    # Identify price index columns
    id_cols   = [year_col, month_col]
    value_cols = [c for c in df_ffpi.columns if c not in id_cols]

    df_ffpi_long = df_ffpi[id_cols + value_cols].melt(
        id_vars=id_cols, var_name="index_name", value_name="index_value"
    )
    df_ffpi_long.rename(columns={year_col: "year", month_col: "month"}, inplace=True)
    df_ffpi_long["year"]  = pd.to_numeric(df_ffpi_long["year"],  errors="coerce")
    df_ffpi_long["month"] = pd.to_numeric(df_ffpi_long["month"], errors="coerce")
    df_ffpi_long["index_value"] = pd.to_numeric(df_ffpi_long["index_value"], errors="coerce")
    df_ffpi_long.dropna(subset=["year", "month", "index_value"], inplace=True)
    df_ffpi_long["year"]  = df_ffpi_long["year"].astype(int)
    df_ffpi_long["month"] = df_ffpi_long["month"].astype(int)
    df_ffpi_long["year_month"] = (
        df_ffpi_long["year"].astype(str) + "-"
        + df_ffpi_long["month"].astype(str).str.zfill(2)
    )
    print(f"FFPI long shape: {df_ffpi_long.shape}")
    print(f"Indices: {df_ffpi_long['index_name'].unique().tolist()}")
else:
    print("Could not parse year/month columns; writing raw table")
    df_ffpi_long = df_ffpi

In [ ]:
write_parquet(df_ffpi_long, f"raw/fao/ffpi/{RUN_DATE}/fao_food_price_index.parquet")

## 2. FAOSTAT Country-Level Food CPI

We pull the **Consumer Prices** domain (CP) from FAOSTAT, filtered to the
food item group. This gives annual country-level food inflation — the predictor
operationalised as `food_price_inflation` in the meta-plan Domain 1.

In [ ]:
print("Pulling FAOSTAT country food CPI (bulk download)...")

# FAOSTAT bulk download: domain CP as CSV zip
FAOSTAT_BULK_URL = f"{FAOSTAT_API_BASE}/bulkdownloads/CP_All_Data_Normalized.zip"

try:
    resp = requests.get(FAOSTAT_BULK_URL, timeout=120, stream=True)
    resp.raise_for_status()
    content = resp.content

    with zipfile.ZipFile(io.BytesIO(content)) as z:
        csv_name = next((n for n in z.namelist() if n.endswith(".csv")), None)
        if csv_name:
            with z.open(csv_name) as f:
                df_cpi_raw = pd.read_csv(f, low_memory=False)
        else:
            raise ValueError("No CSV found inside FAOSTAT zip")

    print(f"FAOSTAT CP raw: {df_cpi_raw.shape}")

    # Normalise and filter to food items
    df_cpi_raw.columns = [c.strip().lower().replace(" ", "_") for c in df_cpi_raw.columns]

    # Keep only food-related item codes (FAOSTAT item 23011 = Food general)
    food_keywords = ["food"]
    item_col = next((c for c in df_cpi_raw.columns if "item" in c and "code" not in c), None)
    if item_col:
        df_food_cpi = df_cpi_raw[
            df_cpi_raw[item_col].str.lower().str.contains("food", na=False)
        ].copy()
    else:
        df_food_cpi = df_cpi_raw.copy()

    print(f"Food CPI rows: {len(df_food_cpi):,}")
    write_parquet(df_food_cpi, f"raw/fao/country_cpi/{RUN_DATE}/faostat_food_cpi.parquet")

except Exception as e:
    print(f"FAOSTAT bulk download failed: {e}")
    print("Tip: download CP_All_Data_Normalized.zip manually from https://www.fao.org/faostat/en/#data/CP")
    df_food_cpi = pd.DataFrame()

## Summary

In [ ]:
print("=" * 55)
print("FAO food price pull complete")
print("=" * 55)
print(f"  Global FFPI       : {len(df_ffpi_long):,} rows")
print(f"  Country food CPI  : {len(df_food_cpi):,} rows")
print()
print("ADLS paths written:")
print(f"  raw/fao/ffpi/{RUN_DATE}/fao_food_price_index.parquet")
print(f"  raw/fao/country_cpi/{RUN_DATE}/faostat_food_cpi.parquet")